In [1]:
import pandas as pd
import time
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import undetected_chromedriver as uc
import re

In [ ]:
def scrape_userbenchmark_gpus():
    url = "https://gpu.userbenchmark.com/"
    
    options = uc.ChromeOptions()
    options.headless = False  
    driver = uc.Chrome(options=options)
    
    all_gpu_data = []
    
    try:
        print(f"Loading {url}...")
        driver.get(url)
        time.sleep(5)  
        
        page_number = 1
        
        while True:
            print(f"Scraping Page {page_number}...")
            
            try:
        
                WebDriverWait(driver, 10).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, "table.mh-td tbody tr"))
                )
            except:
                print("Table not found or took too long to load.")
                break

            soup = BeautifulSoup(driver.page_source, 'html.parser')
            
            tables = soup.find_all('table', class_='mh-td')
            data_table = None
            for tbl in tables:
                tbody = tbl.find('tbody')
                if tbody and len(tbody.find_all('tr')) > 0:
                    data_table = tbl
                    break
                    
            if not data_table:
                print("No data rows found on this page.")
                break
                
            rows = data_table.find('tbody').find_all('tr')
            print(f"Found {len(rows)} rows on this page.")
            
            for row in rows:
                cols = row.find_all('td')
                if len(cols) < 5:
                    continue  
                
                row_data = {}
                
                try:
            
                    links = row.find_all('a')
                    name = None
                    for link in links:
                        link_text = link.text.strip()
                        if len(link_text) > 4 and link_text.lower() not in ["compare", "samples"]:
                            name = link_text
                            break
                            
                    if not name:
                        name = cols[2].text.strip()

                    row_data['Name'] = re.sub(r'Compare|Samples.*', '', name, flags=re.IGNORECASE).strip()

                    text_values = [col.text.strip() for col in cols]
                    
                    def extract_number(text):
                        match = re.search(r'[\d\.,]+', text)
                        return match.group(0) if match else None

                    if len(text_values) >= 8:
                        row_data['Price'] = text_values[-1].replace(',', '').replace('$', '').replace('Rp', '').strip()
                        row_data['Age (Months)'] = extract_number(text_values[-2])
                        row_data['Mkt Share %'] = extract_number(text_values[-3])
                        row_data['Avg Bench %'] = extract_number(text_values[-4])
                        row_data['Value %'] = extract_number(text_values[-5])
                        row_data['User Rating %'] = extract_number(text_values[-6])

                    all_gpu_data.append(row_data)
                    
                except Exception as e:
                    continue

            try:
                next_button = driver.find_element(By.XPATH, "//ul[contains(@class, 'pagination')]//a[contains(., 'Next')]")
                
                if "disabled" in next_button.find_element(By.XPATH, "..").get_attribute("class"):
                    print("Reached the last page.")
                    break
                    
                driver.execute_script("arguments[0].click();", next_button)
                page_number += 1
                time.sleep(3) 
            except Exception as e:
                print("No 'Next' button found, ending pagination.")
                break

    finally:
        driver.quit()
        
    df = pd.DataFrame(all_gpu_data)
    return df

ub_dataframe = scrape_userbenchmark_gpus()

if ub_dataframe is not None and not ub_dataframe.empty:
    ub_dataframe.dropna(subset=['Name'], inplace=True)
    ub_dataframe = ub_dataframe[ub_dataframe['Name'] != ""] 
    
    display(ub_dataframe.head(10))
    
    csv_filename = "userbenchmark_gpus.csv"
    ub_dataframe.to_csv(csv_filename, index=False)
    print(f"\nData successfully saved to {csv_filename} ({len(ub_dataframe)} rows extracted)")
else:
    print("Failed to extract data.")

Loading https://gpu.userbenchmark.com/...
Scraping Page 1...
Found 50 rows on this page.
No 'Next' button found, ending pagination.


,Name,Price,Age (Months),Mkt Share %,Avg Bench %,Value %,User Rating %
0,RTX 5060,524223865th,1199,299,61.246,42.797,105
1,RTX 5060-Ti,585899059th,1299,1.9998,71.456,44.598,103
2,RTX 5070,980620231st,1499,2.9999,98.583,36.793,100
3,RTX 4060-Ti,678319248th,3594,1.0996,60.952,32.889,98
4,RTX 4070,1023253329th,3794,1.2498,8069,28.680,98
5,RTX 4060,616736654th,3495,1.8699,50.948,30.285,97
6,GTX 1660S (Super),306834187th,7878,1.2197,29.927,35.692,97
7,RTX 3070,503886467th,6682,2.4100,61.754,44.798,96
8,RTX 3060,712317745th,6284,3.18100,4238,21.558,96
9,RTX 3080,1117847624th,6782,1.8399,78.971,25.871,94



Data successfully saved to userbenchmark_gpus.csv (50 rows extracted)
